In [36]:
import pandas as pd

# Load the datasets
train_transaction = pd.read_csv('datasets/train_transaction.csv')
train_identity = pd.read_csv('datasets/train_identity.csv')

print("Shape of train_transaction:", train_transaction.shape)
print("Shape of train_identity:", train_identity.shape)

Shape of train_transaction: (590540, 394)
Shape of train_identity: (144233, 41)


In [41]:
# Merge datasets on 'TransactionID'
train_data = train_transaction.merge(train_identity, on='TransactionID', how='left')

print("Shape of train_data:", train_data.shape)

Shape of train_data: (590540, 434)


In [42]:
missing_values_before = train_data.isnull().sum()
print("Missing Values:\n", missing_values_before[missing_values_before > 0])

Missing Values:
 card2           8933
card3           1565
card4           1577
card5           4259
card6           1571
               ...  
id_36         449555
id_37         449555
id_38         449555
DeviceType    449730
DeviceInfo    471874
Length: 414, dtype: int64


In [44]:
# Backup the target column
target = train_data['isFraud']

# Identify columns by type
categorical_cols = train_data.select_dtypes(include='object').columns
numerical_cols = train_data.select_dtypes(include=['int64', 'float64']).columns.drop('isFraud', errors='ignore')

# Fill missing values
# Categorical: replace NaN with 'Missing'
train_data[categorical_cols] = train_data[categorical_cols].fillna("Missing")

# Numerical: replace NaN with median
for col in numerical_cols:
    median_val = train_data[col].median()
    train_data[col] = train_data[col].fillna(median_val)

# Reattach the target column (if needed)
train_data['isFraud'] = target

# Final check
missing_values_after = train_data.isnull().sum()
print("Missing Values after filling:\n", missing_values_after[missing_values_after > 0])

Missing Values after filling:
 Series([], dtype: int64)


In [46]:
from sklearn.model_selection import StratifiedKFold
import pickle

# Export purged data
train_data.to_csv('datasets/train_dataset.csv', index=False)

X = train_data.drop(['isFraud'], axis=1)
y = train_data['isFraud']

print("Total samples:", len(y))
print("Fraud cases:", y.sum())
print("Non-fraud cases:", (y == 0).sum())
print("Fraud ratio:", y.mean())

# Initialize 10-fold stratified CV
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
splits = list(skf.split(X, y))

# Generate and save the 10-fold indices for reuse
with open('10_fold_indices.pkl', 'wb') as f:
    pickle.dump(splits, f)

train_dataset = pd.read_csv('datasets/train_dataset.csv')
print("Shape of train_dataset:", train_dataset.shape)
print("Done")

Total samples: 590540
Fraud cases: 20663
Non-fraud cases: 569877
Fraud ratio: 0.03499000914417313
Shape of train_dataset: (590540, 434)
Done
